# S6E4 Predicting Irrigation Need

## Score: .97068


In [13]:
from pathlib import Path
import os
import time

os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 4))

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 200)


In [14]:
PROJECT_ROOT = Path(".")
DATA_DIR = PROJECT_ROOT / "playground-series-s6e4"
ORIGINAL_PATH = PROJECT_ROOT / "original-data" / "irrigation_prediction.csv"
ANCHOR_SUB_PATH = PROJECT_ROOT / "score97171.csv"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SUB_PATH = PROJECT_ROOT / "submission.csv"

TARGET = "Irrigation_Need"
ID_COL = "id"
SOURCE_COL = "_source"
BASE_SEED = 42
OVERNIGHT_MODE = False
SEED_LIST = [42, 1337, 2026]
N_SPLITS = 5 if OVERNIGHT_MODE else 2
STACKER_SPLITS = 7
STACKER_C_GRID = [
    0.25,
    0.35,
    0.45,
    0.5,
    0.65,
    0.75,
    0.85,
    0.95,
    1.0,
    1.1,
    1.25,
    1.5,
    2.0,
    3.0,
    4.0,
]
TEMPERATURE_GRID = [
    0.64,
    0.68,
    0.72,
    0.76,
    0.8,
    0.84,
    0.88,
    0.92,
    0.96,
    1.0,
    1.04,
    1.08,
    1.12,
    1.16,
]
USE_ORIGINAL_DATA = True
ORIGINAL_ROW_WEIGHT = 0.28
ANCHOR_BLEND_WEIGHT = 0.055
USE_ENGINEERED_FEATURES = True
USE_TARGET_ENCODING = True
USE_CAT2 = True
TE_N_SPLITS = 5
TE_SMOOTHING = 42.0
USE_PSEUDO_LABEL = OVERNIGHT_MODE  # pseudo phase-2 only overnight (~2x train time)
PSEUDO_CONF_MIN = 0.994
PSEUDO_MAX_PER_CLASS = 5000
PSEUDO_ROW_WEIGHT = 0.22
USE_ANCHOR_FALLBACK = False
FALLBACK_CONF_THRESHOLD = 0.60
SUBMISSION_EXTRA_BLENDS = []


if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError(
        f"Expected data files under {DATA_DIR.resolve()}"
    )

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

train_df[SOURCE_COL] = "kaggle"

if USE_ORIGINAL_DATA and ORIGINAL_PATH.exists():
    original_df = pd.read_csv(ORIGINAL_PATH)
    original_df = original_df[[c for c in train_df.columns if c not in [ID_COL, SOURCE_COL]]]
    original_df[SOURCE_COL] = "original"
    train_df = pd.concat([train_df, original_df], axis=0, ignore_index=True)
    print("original shape:", original_df.shape)
else:
    print("original data disabled or missing")

print("train shape:", train_df.shape)
print("test shape :", test_df.shape)
train_df.head()


original shape: (10000, 21)
train shape: (640000, 22)
test shape : (270000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need,_source
0,0.0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low,kaggle
1,1.0,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low,kaggle
2,2.0,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low,kaggle
3,3.0,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium,kaggle
4,4.0,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low,kaggle


In [15]:
display(train_df[TARGET].value_counts(dropna=False))
print("\nTarget distribution (%):")
display((train_df[TARGET].value_counts(normalize=True) * 100).round(2))

missing_train = train_df.isna().mean().sort_values(ascending=False)
missing_test = test_df.isna().mean().sort_values(ascending=False)

print("\nTop missing-rate features (train):")
display(missing_train.head(10))
print("\nTop missing-rate features (test):")
display(missing_test.head(10))

Irrigation_Need
Low       375781
Medium    242874
High       21345
Name: count, dtype: int64


Target distribution (%):


Irrigation_Need
Low       58.72
Medium    37.95
High       3.34
Name: proportion, dtype: float64


Top missing-rate features (train):


id                         0.015625
Soil_Type                  0.000000
Soil_pH                    0.000000
Soil_Moisture              0.000000
Organic_Carbon             0.000000
Electrical_Conductivity    0.000000
Temperature_C              0.000000
Humidity                   0.000000
Rainfall_mm                0.000000
Sunlight_Hours             0.000000
dtype: float64


Top missing-rate features (test):


id                         0.0
Soil_Type                  0.0
Soil_pH                    0.0
Soil_Moisture              0.0
Organic_Carbon             0.0
Electrical_Conductivity    0.0
Temperature_C              0.0
Humidity                   0.0
Rainfall_mm                0.0
Sunlight_Hours             0.0
dtype: float64

In [16]:



def add_engineered_features(df):
    out = df.copy()
    eps = 1e-6
    out["feat_rain_per_moisture"] = out["Rainfall_mm"] / (out["Soil_Moisture"] + eps)
    out["feat_humid_temp"] = out["Humidity"] * out["Temperature_C"] / 100.0
    out["feat_prev_per_area"] = out["Previous_Irrigation_mm"] / (out["Field_Area_hectare"] + eps)
    out["feat_sun_wind"] = out["Sunlight_Hours"] * out["Wind_Speed_kmh"] / 100.0
    out["feat_oc_ph"] = out["Organic_Carbon"] * out["Soil_pH"]
    out["feat_net_water_mm"] = out["Rainfall_mm"] - out["Previous_Irrigation_mm"]
    out["feat_rain_per_sun"] = out["Rainfall_mm"] / (out["Sunlight_Hours"] + eps)
    out["feat_drying_index"] = (
        out["Temperature_C"] * out["Wind_Speed_kmh"] / (out["Humidity"] + eps)
    )
    out["feat_ec_over_moisture"] = out["Electrical_Conductivity"] / (
        out["Soil_Moisture"] + eps
    )
    out["feat_moisture_ph"] = out["Soil_Moisture"] * out["Soil_pH"] / 100.0
    out["feat_irrig_share"] = out["Previous_Irrigation_mm"] / (
        out["Rainfall_mm"] + out["Previous_Irrigation_mm"] + eps
    )
    out["feat_log1p_rain"] = np.log1p(out["Rainfall_mm"].clip(lower=0))
    out["feat_log1p_prev"] = np.log1p(out["Previous_Irrigation_mm"].clip(lower=0))
    out["feat_temp_div_humid"] = out["Temperature_C"] / (out["Humidity"] + eps)
    out["feat_wind_div_rain"] = out["Wind_Speed_kmh"] / (out["Rainfall_mm"] + eps)
    out["feat_moisture_sqrt"] = np.sqrt(out["Soil_Moisture"].clip(lower=0))
    out["feat_ph_moisture"] = out["Soil_pH"] * out["Soil_Moisture"] / 100.0
    out["feat_vpd_proxy"] = out["Temperature_C"] * (1.0 - out["Humidity"] / 100.0)
    return out


def _fit_te_map(cat_series, y_series, classes, priors, m):
    df = pd.DataFrame({"c": cat_series.values, "y": y_series.values})
    te_map = {}
    for cval, grp in df.groupby("c", sort=False):
        n = len(grp)
        counts = grp["y"].value_counts()
        arr = np.array([counts.get(cl, 0) for cl in classes], dtype=np.float64)
        te_map[cval] = (arr + m * priors) / (n + m)
    return te_map, priors.copy()


def _apply_te_rows(series, te_map, default, n_classes):
    out = np.zeros((len(series), n_classes), dtype=np.float64)
    vals = series.values
    for i in range(len(series)):
        out[i] = te_map.get(vals[i], default)
    return out


def add_multiclass_target_encoding_oof(
    train_df, test_df, target_col, cat_cols, seed, n_splits, m
):
    y = train_df[target_col]
    classes_local = sorted(y.unique().tolist())
    n_classes = len(classes_local)
    priors = (
        y.value_counts(normalize=True).reindex(classes_local).fillna(0).values.astype(np.float64)
    )
    oof_mat = np.zeros((len(train_df), len(cat_cols) * n_classes), dtype=np.float64)
    col_off = {c: i * n_classes for i, c in enumerate(cat_cols)}
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, va_idx in skf.split(train_df, y):
        y_tr = y.iloc[tr_idx]
        for col in cat_cols:
            te_map, default = _fit_te_map(
                train_df[col].iloc[tr_idx], y_tr, classes_local, priors, m
            )
            block = _apply_te_rows(
                train_df[col].iloc[va_idx], te_map, default, n_classes
            )
            sl = slice(col_off[col], col_off[col] + n_classes)
            oof_mat[va_idx, sl] = block
    test_mat = np.zeros((len(test_df), len(cat_cols) * n_classes), dtype=np.float64)
    for col in cat_cols:
        te_map, default = _fit_te_map(train_df[col], y, classes_local, priors, m)
        test_mat[:, col_off[col] : col_off[col] + n_classes] = _apply_te_rows(
            test_df[col], te_map, default, n_classes
        )
    for i, col in enumerate(cat_cols):
        for k, cls in enumerate(classes_local):
            name = "te_" + str(col).replace(" ", "_") + "_" + str(cls).replace(" ", "_")
            train_df[name] = oof_mat[:, i * n_classes + k]
            test_df[name] = test_mat[:, i * n_classes + k]
    print(
        f"OOF target encoding: +{len(cat_cols) * n_classes} cols (splits={n_splits}, m={m})"
    )
    return train_df, test_df


if USE_ENGINEERED_FEATURES:
    train_df = add_engineered_features(train_df)
    test_df = add_engineered_features(test_df)

pre_cols = [c for c in train_df.columns if c not in [TARGET, ID_COL, SOURCE_COL]]
_cat_for_te = [c for c in pre_cols if train_df[c].dtype == "object"]
if USE_TARGET_ENCODING and len(_cat_for_te) > 0:
    train_df, test_df = add_multiclass_target_encoding_oof(
        train_df,
        test_df,
        TARGET,
        _cat_for_te,
        BASE_SEED,
        TE_N_SPLITS,
        TE_SMOOTHING,
    )

TRAIN_DF_BASE = train_df.copy()

y = train_df[TARGET].copy()
source_flag = train_df[SOURCE_COL].copy()
X = train_df.drop(columns=[TARGET]).copy()
X_test = test_df.copy()

feature_cols = [c for c in X.columns if c not in [ID_COL, SOURCE_COL]]
X = X[feature_cols]
X_test = X_test[feature_cols]

cat_cols = [c for c in feature_cols if X[c].dtype == "object"]
num_cols = [c for c in feature_cols if c not in cat_cols]

print(f"Total features: {len(feature_cols)}")
print(f"Categorical   : {len(cat_cols)} -> {cat_cols}")
print(f"Numerical     : {len(num_cols)}")
print("Source distribution:")
display(source_flag.value_counts())


OOF target encoding: +24 cols (splits=5, m=42.0)
Total features: 61
Categorical   : 8 -> ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
Numerical     : 53
Source distribution:


_source
kaggle      630000
original     10000
Name: count, dtype: int64

In [17]:
classes = sorted(TRAIN_DF_BASE[TARGET].unique().tolist())
class_to_idx = {label: idx for idx, label in enumerate(classes)}
idx_to_class = {v: k for k, v in class_to_idx.items()}

if OVERNIGHT_MODE:
    cat_iters, cat_lr, cat_depth, cat_es, cat_verbose = 1200, 0.04, 8, 120, 200
    lgb_est, lgb_lr, lgb_leaves, lgb_es = 1200, 0.035, 96, 120
    xgb_est, xgb_lr, xgb_depth, xgb_es = 1200, 0.035, 8, 120
    cat2_iters, cat2_lr, cat2_depth, cat2_es, cat2_verbose = 900, 0.05, 6, 100, 150
else:
    cat_iters, cat_lr, cat_depth, cat_es, cat_verbose = 700, 0.06, 7, 80, 100
    lgb_est, lgb_lr, lgb_leaves, lgb_es = 800, 0.04, 63, 80
    xgb_est, xgb_lr, xgb_depth, xgb_es = 800, 0.04, 7, 80
    cat2_iters, cat2_lr, cat2_depth, cat2_es, cat2_verbose = 420, 0.085, 5, 75, 80


def build_meta_features(*parts):
    eps = 1e-15
    parts = tuple(parts)
    blocks = [np.hstack(parts)]
    for p in parts:
        blocks.append(np.log(np.clip(p, eps, 1.0)))
    for p in parts:
        ent = -(p * np.log(np.clip(p, eps, 1.0))).sum(axis=1, keepdims=True)
        blocks.append(ent)
    for p in parts:
        blocks.append(np.max(p, axis=1, keepdims=True))
    for p in parts:
        ps = np.sort(p, axis=1)
        margin = (ps[:, -1] - ps[:, -2]).reshape(-1, 1)
        blocks.append(margin)
    return np.hstack(blocks)


def build_pseudo_augmented_train(train_df_base, test_df, test_probs):
    rng = np.random.RandomState(BASE_SEED + 99)
    conf = test_probs.max(axis=1)
    pred_i = np.argmax(test_probs, axis=1)
    keep_idx = []
    for cls_idx in range(len(classes)):
        mask = (conf >= PSEUDO_CONF_MIN) & (pred_i == cls_idx)
        ix = np.where(mask)[0]
        if len(ix) > PSEUDO_MAX_PER_CLASS:
            ix = rng.choice(ix, PSEUDO_MAX_PER_CLASS, replace=False)
        keep_idx.extend(ix.tolist())
    keep_idx = np.unique(np.array(keep_idx, dtype=int))
    pseudo = test_df.iloc[keep_idx].copy()
    pseudo[TARGET] = [idx_to_class[pred_i[i]] for i in keep_idx]
    pseudo[SOURCE_COL] = "pseudo"
    out = pd.concat([train_df_base, pseudo], axis=0, ignore_index=True)
    print(
        f"Pseudo-label: added {len(pseudo)} rows "
        f"(conf>={PSEUDO_CONF_MIN}, cap={PSEUDO_MAX_PER_CLASS}/class)"
    )
    return out


def temper_probs(p, T):
    z = np.log(np.clip(p, 1e-15, 1.0))
    zs = z / T
    ex = np.exp(zs - zs.max(axis=1, keepdims=True))
    return ex / ex.sum(axis=1, keepdims=True)


def run_training_phase(train_df_run, phase_name):
    run_start = time.perf_counter()
    print(f"\n{'='*20} {phase_name} {'='*20}")
    print(
        f"Config: OVERNIGHT_MODE={OVERNIGHT_MODE} | seeds={len(SEED_LIST)} | "
        f"n_splits={N_SPLITS} | stacker_splits={STACKER_SPLITS} | USE_CAT2={USE_CAT2} | "
        f"C_grid={len(STACKER_C_GRID)} | T_grid={len(TEMPERATURE_GRID)}"
    )

    y = train_df_run[TARGET].copy()
    source_flag = train_df_run[SOURCE_COL].copy()
    X = train_df_run.drop(columns=[TARGET]).copy()
    X_test = test_df.copy()
    feature_cols = [c for c in X.columns if c not in [ID_COL, SOURCE_COL]]
    X = X[feature_cols]
    X_test = X_test[feature_cols]
    cat_cols = [c for c in feature_cols if X[c].dtype == "object"]

    y_idx = y.map(class_to_idx).astype(int)
    class_counts = y.value_counts()
    class_weight_label = {
        cls: len(y) / (len(classes) * class_counts[cls])
        for cls in classes
    }
    class_weight_idx = {
        class_to_idx[cls]: weight
        for cls, weight in class_weight_label.items()
    }
    source_weight_map = {
        "kaggle": 1.0,
        "original": ORIGINAL_ROW_WEIGHT,
        "pseudo": PSEUDO_ROW_WEIGHT,
    }

    oof_cat = np.zeros((len(X), len(classes)), dtype=np.float64)
    oof_cat2 = np.zeros((len(X), len(classes)), dtype=np.float64)
    oof_lgb = np.zeros((len(X), len(classes)), dtype=np.float64)
    oof_xgb = np.zeros((len(X), len(classes)), dtype=np.float64)

    test_cat = np.zeros((len(X_test), len(classes)), dtype=np.float64)
    test_cat2 = np.zeros((len(X_test), len(classes)), dtype=np.float64)
    test_lgb = np.zeros((len(X_test), len(classes)), dtype=np.float64)
    test_xgb = np.zeros((len(X_test), len(classes)), dtype=np.float64)

    fold_scores = []

    for seed_idx, seed in enumerate(SEED_LIST, start=1):
        seed_start = time.perf_counter()
        print(f"\n######## Seed {seed_idx}/{len(SEED_LIST)} (seed={seed}) ########")

        cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

        seed_test_cat = np.zeros((len(X_test), len(classes)), dtype=np.float64)
        seed_test_cat2 = np.zeros((len(X_test), len(classes)), dtype=np.float64)
        seed_test_lgb = np.zeros((len(X_test), len(classes)), dtype=np.float64)
        seed_test_xgb = np.zeros((len(X_test), len(classes)), dtype=np.float64)

        for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
            fold_start = time.perf_counter()
            print(f"\n=== Seed {seed} | Fold {fold}/{N_SPLITS} started ===")

            X_train, X_valid = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
            y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
            y_train_idx, y_valid_idx = y_idx.iloc[train_idx], y_idx.iloc[valid_idx]
            source_train = source_flag.iloc[train_idx]

            for col in cat_cols:
                X_train[col] = X_train[col].astype("category")
                X_valid[col] = X_valid[col].astype("category")

            X_test_fold = X_test.copy()
            for col in cat_cols:
                X_test_fold[col] = X_test_fold[col].astype("category")

            sample_weight = (
                y_train.map(class_weight_label).values
                * source_train.map(source_weight_map).values
            )

            cat_model = CatBoostClassifier(
                iterations=cat_iters,
                learning_rate=cat_lr,
                depth=cat_depth,
                loss_function="MultiClass",
                eval_metric="TotalF1",
                random_seed=seed + fold,
                verbose=cat_verbose,
                thread_count=-1,
                class_weights=class_weight_label,
            )

            lgb_model = LGBMClassifier(
                n_estimators=lgb_est,
                learning_rate=lgb_lr,
                num_leaves=lgb_leaves,
                min_child_samples=40,
                subsample=0.9,
                colsample_bytree=0.9,
                reg_lambda=1.0,
                reg_alpha=0.0,
                objective="multiclass",
                random_state=seed + fold,
                class_weight=class_weight_idx,
                verbosity=-1,
            )

            xgb_model = XGBClassifier(
                n_estimators=xgb_est,
                learning_rate=xgb_lr,
                max_depth=xgb_depth,
                min_child_weight=3,
                gamma=0.04,
                subsample=0.9,
                colsample_bytree=0.9,
                reg_lambda=1.0,
                objective="multi:softprob",
                num_class=len(classes),
                eval_metric="mlogloss",
                tree_method="hist",
                enable_categorical=True,
                random_state=seed + fold,
                n_jobs=-1,
            )

            print(f"Seed {seed} Fold {fold}: training CatBoost...")
            cat_model.fit(
                X_train,
                y_train,
                sample_weight=sample_weight,
                cat_features=cat_cols,
                eval_set=(X_valid, y_valid),
                use_best_model=True,
                early_stopping_rounds=cat_es,
            )

            if USE_CAT2:
                cat2_model = CatBoostClassifier(
                    iterations=cat2_iters,
                    learning_rate=cat2_lr,
                    depth=cat2_depth,
                    loss_function="MultiClass",
                    eval_metric="TotalF1",
                    random_seed=seed + fold + 777,
                    verbose=cat2_verbose,
                    thread_count=-1,
                    class_weights=class_weight_label,
                )
                print(f"Seed {seed} Fold {fold}: training CatBoost-2...")
                cat2_model.fit(
                    X_train,
                    y_train,
                    sample_weight=sample_weight,
                    cat_features=cat_cols,
                    eval_set=(X_valid, y_valid),
                    use_best_model=True,
                    early_stopping_rounds=cat2_es,
                )

            print(f"Seed {seed} Fold {fold}: training LightGBM...")
            lgb_model.fit(
                X_train,
                y_train_idx,
                sample_weight=sample_weight,
                eval_set=[(X_valid, y_valid_idx)],
                eval_metric="multi_logloss",
                callbacks=[early_stopping(stopping_rounds=lgb_es), log_evaluation(period=0)],
            )

            print(f"Seed {seed} Fold {fold}: training XGBoost...")
            xgb_model.fit(
                X_train,
                y_train_idx,
                eval_set=[(X_valid, y_valid_idx)],
                sample_weight=sample_weight,
                verbose=False,
            )
            print(f"Seed {seed} Fold {fold}: model training complete")

            cat_valid = cat_model.predict_proba(X_valid)
            if USE_CAT2:
                cat2_valid = cat2_model.predict_proba(X_valid)
            else:
                cat2_valid = None
            lgb_valid = lgb_model.predict_proba(X_valid)
            xgb_valid = xgb_model.predict_proba(X_valid)

            oof_cat[valid_idx] += cat_valid / len(SEED_LIST)
            if USE_CAT2:
                oof_cat2[valid_idx] += cat2_valid / len(SEED_LIST)
            oof_lgb[valid_idx] += lgb_valid / len(SEED_LIST)
            oof_xgb[valid_idx] += xgb_valid / len(SEED_LIST)

            cat_pred = np.argmax(cat_valid, axis=1)
            lgb_pred = np.argmax(lgb_valid, axis=1)
            xgb_pred = np.argmax(xgb_valid, axis=1)

            cat_score = balanced_accuracy_score(y_valid, [idx_to_class[i] for i in cat_pred])
            lgb_score = balanced_accuracy_score(y_valid, [idx_to_class[i] for i in lgb_pred])
            xgb_score = balanced_accuracy_score(y_valid, [idx_to_class[i] for i in xgb_pred])
            if USE_CAT2:
                cat2_pred = np.argmax(cat2_valid, axis=1)
                cat2_score = balanced_accuracy_score(
                    y_valid, [idx_to_class[i] for i in cat2_pred]
                )
                print(
                    f"Seed {seed} Fold {fold}: cat={cat_score:.6f} cat2={cat2_score:.6f} "
                    f"lgb={lgb_score:.6f} xgb={xgb_score:.6f}"
                )
            else:
                print(
                    f"Seed {seed} Fold {fold}: cat={cat_score:.6f} "
                    f"lgb={lgb_score:.6f} xgb={xgb_score:.6f}"
                )

            if USE_CAT2:
                blend_equal = (cat_valid + cat2_valid + lgb_valid + xgb_valid) / 4.0
            else:
                blend_equal = (cat_valid + lgb_valid + xgb_valid) / 3.0
            blend_pred = np.argmax(blend_equal, axis=1)
            fold_score = balanced_accuracy_score(y_valid, [idx_to_class[i] for i in blend_pred])
            fold_scores.append(fold_score)
            fold_minutes = (time.perf_counter() - fold_start) / 60.0
            print(f"Seed {seed} Fold {fold}: equal-blend balanced_accuracy = {fold_score:.6f}")
            print(f"Seed {seed} Fold {fold}: completed in {fold_minutes:.2f} minutes")

            seed_test_cat += cat_model.predict_proba(X_test_fold)
            if USE_CAT2:
                seed_test_cat2 += cat2_model.predict_proba(X_test_fold)
            seed_test_lgb += lgb_model.predict_proba(X_test_fold)
            seed_test_xgb += xgb_model.predict_proba(X_test_fold)

        test_cat += seed_test_cat / N_SPLITS
        test_cat2 += seed_test_cat2 / N_SPLITS
        test_lgb += seed_test_lgb / N_SPLITS
        test_xgb += seed_test_xgb / N_SPLITS

        print(
            f"Seed {seed} finished in {(time.perf_counter() - seed_start) / 60.0:.2f} minutes"
        )

    test_cat /= len(SEED_LIST)
    test_cat2 /= len(SEED_LIST)
    test_lgb /= len(SEED_LIST)
    test_xgb /= len(SEED_LIST)

    print("\nAll folds done. Building meta-features + cross-fitted OOF stacking...")

    if USE_CAT2:
        meta_oof = build_meta_features(oof_cat, oof_cat2, oof_lgb, oof_xgb)
        meta_test = build_meta_features(test_cat, test_cat2, test_lgb, test_xgb)
    else:
        meta_oof = build_meta_features(oof_cat, oof_lgb, oof_xgb)
        meta_test = build_meta_features(test_cat, test_lgb, test_xgb)
    print(f"Meta feature columns: {meta_oof.shape[1]}")

    best_stack_score = -1.0
    best_stack_c = None
    best_stack_proba_oof = None
    best_stack_proba_test = None

    stack_cv = StratifiedKFold(n_splits=STACKER_SPLITS, shuffle=True, random_state=BASE_SEED)

    for c in STACKER_C_GRID:
        print(f"Stacking search: cross-fitting LogisticRegression C={c}")

        cv_meta_oof = np.zeros((len(meta_oof), len(classes)), dtype=np.float64)
        cv_meta_test = np.zeros((len(meta_test), len(classes)), dtype=np.float64)

        for stack_fold, (st_tr_idx, st_va_idx) in enumerate(stack_cv.split(meta_oof, y_idx), start=1):
            X_st_tr, X_st_va = meta_oof[st_tr_idx], meta_oof[st_va_idx]
            y_st_tr = y_idx.iloc[st_tr_idx]

            stacker = Pipeline(
                [
                    ("scaler", StandardScaler()),
                    (
                        "clf",
                        LogisticRegression(
                            C=c,
                            max_iter=800,
                            solver="lbfgs",
                            class_weight="balanced",
                        ),
                    ),
                ]
            )
            stacker.fit(X_st_tr, y_st_tr)

            cv_meta_oof[st_va_idx] = stacker.predict_proba(X_st_va)
            cv_meta_test += stacker.predict_proba(meta_test)

            if stack_fold % 2 == 0:
                print(f"Stacking C={c}: finished meta-fold {stack_fold}/{STACKER_SPLITS}")

        cv_meta_test /= STACKER_SPLITS

        pred_oof = np.argmax(cv_meta_oof, axis=1)
        pred_oof_labels = [idx_to_class[i] for i in pred_oof]
        score = balanced_accuracy_score(y, pred_oof_labels)
        print(f"Stacking C={c}: cross-fitted OOF balanced_accuracy = {score:.6f}")

        if score > best_stack_score:
            best_stack_score = score
            best_stack_c = c
            best_stack_proba_oof = cv_meta_oof
            best_stack_proba_test = cv_meta_test

    print(f"Best stacker C: {best_stack_c} with OOF score {best_stack_score:.6f}")

    print("Stacking: LDA meta-learner (OOF) + blend with LR...")
    lda_oof = np.zeros((len(meta_oof), len(classes)), dtype=np.float64)
    lda_test = np.zeros((len(meta_test), len(classes)), dtype=np.float64)
    for stack_fold, (st_tr_idx, st_va_idx) in enumerate(stack_cv.split(meta_oof, y_idx), start=1):
        sc_lda = StandardScaler()
        Xtr = sc_lda.fit_transform(meta_oof[st_tr_idx])
        Xva = sc_lda.transform(meta_oof[st_va_idx])
        Xt = sc_lda.transform(meta_test)
        lda = LinearDiscriminantAnalysis(solver="svd")
        lda.fit(Xtr, y_idx.iloc[st_tr_idx])
        lda_oof[st_va_idx] = lda.predict_proba(Xva)
        lda_test += lda.predict_proba(Xt)
    lda_test /= STACKER_SPLITS

    best_meta_w = 0.0
    best_meta_blend_score = -1.0
    for w in np.linspace(0.0, 1.0, 41):
        blend_oof = (1.0 - w) * best_stack_proba_oof + w * lda_oof
        pred_lbl = [idx_to_class[i] for i in np.argmax(blend_oof, axis=1)]
        sc = balanced_accuracy_score(y, pred_lbl)
        if sc > best_meta_blend_score:
            best_meta_blend_score = sc
            best_meta_w = w

    best_stack_proba_oof = (1.0 - best_meta_w) * best_stack_proba_oof + best_meta_w * lda_oof
    best_stack_proba_test = (1.0 - best_meta_w) * best_stack_proba_test + best_meta_w * lda_test
    best_stack_score = best_meta_blend_score
    print(
        f"Meta LR+LDA blend w={best_meta_w:.2f} OOF balanced_accuracy={best_meta_blend_score:.6f}"
    )

    print("Searching temperature scaling on stacker OOF...")

    best_T = 1.0
    best_temp_score = -1.0
    for T in TEMPERATURE_GRID:
        tp = temper_probs(best_stack_proba_oof, T)
        pred_lbl = [idx_to_class[i] for i in np.argmax(tp, axis=1)]
        sc = balanced_accuracy_score(y, pred_lbl)
        if sc > best_temp_score:
            best_temp_score = sc
            best_T = T

    print(f"Best temperature T={best_T} OOF balanced_accuracy={best_temp_score:.6f}")

    oof_after_temp = temper_probs(best_stack_proba_oof, best_T)
    test_after_temp = temper_probs(best_stack_proba_test, best_T)

    print("Starting class-bias tuning...")

    best_bias_score = -1.0
    best_bias = np.zeros(len(classes), dtype=np.float64)

    bias_checks = 0
    for b_low in np.arange(-0.06, 0.061, 0.01):
        for b_med in np.arange(-0.06, 0.061, 0.01):
            for b_high in np.arange(-0.06, 0.061, 0.01):
                bias = np.array([b_low, b_med, b_high], dtype=np.float64)
                biased_oof = oof_after_temp + bias
                pred_idx = np.argmax(biased_oof, axis=1)
                pred_labels = [idx_to_class[i] for i in pred_idx]
                score = balanced_accuracy_score(y, pred_labels)
                bias_checks += 1
                if bias_checks % 200 == 0:
                    print(f"Bias search progress: checked {bias_checks} combos")
                if score > best_bias_score:
                    best_bias_score = score
                    best_bias = bias.copy()

    oof_best = oof_after_temp + best_bias
    oof_best_pred = np.argmax(oof_best, axis=1)
    oof_best_pred = [idx_to_class[i] for i in oof_best_pred]

    test_proba = test_after_temp + best_bias

    if USE_CAT2:
        base_oof = (oof_cat + oof_cat2 + oof_lgb + oof_xgb) / 4.0
        base_test = (test_cat + test_cat2 + test_lgb + test_xgb) / 4.0
    else:
        base_oof = (oof_cat + oof_lgb + oof_xgb) / 3.0
        base_test = (test_cat + test_lgb + test_xgb) / 3.0

    best_mix_alpha = 0.0
    best_mix_score = -1.0
    best_mix_oof = oof_best.copy()
    for alpha in list(np.linspace(0.0, 0.45, 10)):
        mix_oof = (1.0 - alpha) * oof_best + alpha * base_oof
        mix_pred = [idx_to_class[i] for i in np.argmax(mix_oof, axis=1)]
        mix_score = balanced_accuracy_score(y, mix_pred)
        if mix_score > best_mix_score:
            best_mix_score = mix_score
            best_mix_alpha = alpha
            best_mix_oof = mix_oof

    oof_best = best_mix_oof
    test_proba = (1.0 - best_mix_alpha) * test_proba + best_mix_alpha * base_test

    cv_score = balanced_accuracy_score(y, [idx_to_class[i] for i in np.argmax(oof_best, axis=1)])
    print(f"OOF safety-mix alpha: {best_mix_alpha:.2f} | score={best_mix_score:.6f}")
    print("\nCV balanced accuracy (OOF):", round(cv_score, 6))
    print("Fold equal-blend scores:", [round(s, 6) for s in fold_scores])
    print("Mean equal-blend score:", round(float(np.mean(fold_scores)), 6))
    print(f"Best OOF score from stacking search: {best_stack_score:.6f}")
    print(f"Best temperature used: {best_T}")
    bias_text = ", ".join(
        [f"{cls}={offset:+.3f}" for cls, offset in zip(classes, best_bias)]
    )
    print(f"Best class bias offsets: {bias_text}")
    print(f"Best OOF score after bias tuning: {best_bias_score:.6f}")
    print(f"Total bias combos checked: {bias_checks}")
    print(f"Phase run time: {(time.perf_counter() - run_start) / 60.0:.2f} minutes")

    oof_pred_labels = [idx_to_class[i] for i in np.argmax(oof_best, axis=1)]
    cm_counts = confusion_matrix(y, oof_pred_labels, labels=classes)
    cm_counts_df = pd.DataFrame(cm_counts, index=classes, columns=classes)
    row_sums = cm_counts.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1, row_sums)
    cm_row_pct_df = pd.DataFrame(
        np.round(cm_counts / row_sums, 4),
        index=classes,
        columns=classes,
    )

    return {
        "test_proba": test_proba,
        "test_after_temp": test_after_temp,
        "cv_score": cv_score,
        "oof_bias_score": best_bias_score,
        "oof_pred": oof_pred_labels,
        "y_true": y,
        "cm_counts": cm_counts_df,
        "cm_row_pct": cm_row_pct_df,
    }


total_t0 = time.perf_counter()
print("Starting ensemble CV run...")
print(
    f"ORIGINAL_ROW_WEIGHT={ORIGINAL_ROW_WEIGHT} | "
    f"USE_PSEUDO_LABEL={USE_PSEUDO_LABEL} | "
    f"PSEUDO_CONF_MIN={PSEUDO_CONF_MIN}"
)

out1 = run_training_phase(TRAIN_DF_BASE.copy(), "phase1")
final_out = out1
test_proba = out1["test_proba"]

if USE_PSEUDO_LABEL:
    train_aug = build_pseudo_augmented_train(TRAIN_DF_BASE, test_df, out1["test_after_temp"])
    out2 = run_training_phase(train_aug, "phase2")
    final_out = out2
    test_proba = out2["test_proba"]

print(f"\nTotal wall time: {(time.perf_counter() - total_t0) / 60.0:.2f} minutes")
print("\nOOF confusion matrix (counts):")
display(final_out["cm_counts"])
print("\nOOF confusion matrix (row-normalized):")
display(final_out["cm_row_pct"])


Starting ensemble CV run...
ORIGINAL_ROW_WEIGHT=0.28 | USE_PSEUDO_LABEL=False | PSEUDO_CONF_MIN=0.994

==================== phase1 ====================
Config: OVERNIGHT_MODE=False | seeds=3 | n_splits=2 | stacker_splits=7 | USE_CAT2=True | C_grid=15 | T_grid=14

######## Seed 1/3 (seed=42) ########

=== Seed 42 | Fold 1/2 started ===
Seed 42 Fold 1: training CatBoost...
0:	learn: 0.9348996	test: 0.6954932	best: 0.6954932 (0)	total: 875ms	remaining: 10m 11s
100:	learn: 0.9764428	test: 0.9163622	best: 0.9163622 (100)	total: 1m 11s	remaining: 7m 4s
200:	learn: 0.9800298	test: 0.9223066	best: 0.9223066 (200)	total: 2m 27s	remaining: 6m 7s
300:	learn: 0.9838791	test: 0.9301116	best: 0.9301116 (300)	total: 3m 42s	remaining: 4m 54s
400:	learn: 0.9866409	test: 0.9378359	best: 0.9378359 (400)	total: 4m 49s	remaining: 3m 35s
500:	learn: 0.9888072	test: 0.9442674	best: 0.9442674 (500)	total: 5m 42s	remaining: 2m 15s
600:	learn: 0.9902491	test: 0.9489984	best: 0.9489984 (600)	total: 6m 36s	remain

,High,Low,Medium
High,20463,0,882
Low,2,374104,1675
Medium,4264,5224,233386



OOF confusion matrix (row-normalized):


,High,Low,Medium
High,0.9587,0.0000,0.0413
Low,0.0000,0.9955,0.0045
Medium,0.0176,0.0215,0.9609


In [18]:
test_proba_blend = test_proba.copy()
if ANCHOR_BLEND_WEIGHT > 0 and ANCHOR_SUB_PATH.exists():
    anchor_df = pd.read_csv(ANCHOR_SUB_PATH)[[ID_COL, TARGET]].copy()
    anchor_map = anchor_df.set_index(ID_COL)[TARGET]
    anchor_labels = test_df[ID_COL].map(anchor_map)
    if anchor_labels.isna().any():
        raise ValueError("Anchor blend file is missing ids from test set")
    anchor_idx = anchor_labels.map(class_to_idx).astype(int).values
    anchor_oh = np.zeros_like(test_proba_blend, dtype=np.float64)
    anchor_oh[np.arange(len(anchor_oh)), anchor_idx] = 1.0
    w = float(ANCHOR_BLEND_WEIGHT)
    test_proba_blend = (1.0 - w) * test_proba_blend + w * anchor_oh
    print(f"Anchor probability blend applied (weight={w})")
else:
    print("Anchor probability blend skipped (weight=0 or file missing)")

for path, wt in SUBMISSION_EXTRA_BLENDS:
    if wt <= 0:
        continue
    if not path.exists():
        print(f"Extra submission blend skipped (missing): {path}")
        continue
    sub_df = pd.read_csv(path)[[ID_COL, TARGET]].copy()
    amap = sub_df.set_index(ID_COL)[TARGET]
    alabels = test_df[ID_COL].map(amap)
    if alabels.isna().any():
        raise ValueError(f"Extra blend file missing test ids: {path}")
    aidx = alabels.map(class_to_idx).astype(int).values
    aoh = np.zeros_like(test_proba_blend, dtype=np.float64)
    aoh[np.arange(len(aoh)), aidx] = 1.0
    w = float(wt)
    test_proba_blend = (1.0 - w) * test_proba_blend + w * aoh
    print(f"Extra submission blend: {path.name} weight={w}")

test_conf = np.max(test_proba_blend, axis=1)
test_pred_idx = np.argmax(test_proba_blend, axis=1)
test_pred = pd.Series([idx_to_class[i] for i in test_pred_idx], index=test_df.index)

if USE_ANCHOR_FALLBACK and ANCHOR_SUB_PATH.exists():
    anchor_df = pd.read_csv(ANCHOR_SUB_PATH)
    anchor_df = anchor_df[[ID_COL, TARGET]].copy()
    anchor_map = anchor_df.set_index(ID_COL)[TARGET]
    anchor_labels = test_df[ID_COL].map(anchor_map)

    if anchor_labels.isna().any():
        raise ValueError("Anchor submission is missing ids from test set")

    use_anchor_mask = test_conf < FALLBACK_CONF_THRESHOLD
    test_pred.loc[use_anchor_mask] = anchor_labels.loc[use_anchor_mask]
    print(
        f"Anchor fallback applied to {int(use_anchor_mask.sum())} rows "
        f"(threshold={FALLBACK_CONF_THRESHOLD})"
    )
else:
    print("Anchor fallback disabled or anchor file missing")

_sample_cols = pd.read_csv(DATA_DIR / "sample_submission.csv", nrows=0).columns.tolist()
_sub_id_name, _sub_target_name = _sample_cols[0], _sample_cols[1]
submission = pd.DataFrame(
    {
        _sub_id_name: test_df[ID_COL].astype("int64").to_numpy(),
        _sub_target_name: test_pred.to_numpy(),
    }
)
assert len(submission) == len(test_df), "submission length must match test"
submission.to_csv(SUB_PATH, index=False, encoding="utf-8", lineterminator="\n")

print("Saved:", SUB_PATH.resolve())
print("Submission columns (must match sample):", list(submission.columns))
submission.head()


Anchor probability blend applied (weight=0.055)
Anchor fallback disabled or anchor file missing
Saved: C:\Users\ol1v3_7dwns5u\OneDrive\Desktop\ACTIVE\s6e4-predicting-irrigation-need\submission.csv
Submission columns (must match sample): ['id', 'Irrigation_Need']


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low
